In [20]:
import h5py
import numpy as np

# Load the PCM dataset
data_path = r"C:\Users\risha\Desktop\pcm_project\pcm_2d_dataset.h5"

with h5py.File(data_path, "r") as f:
    temperature = f["T"][:]
    liquid_fraction = f["lf"][:]
    velocity_x = f["ux"][:]
    velocity_y = f["uy"][:]
    
    time_points = f["t"][:]
    x_coords = f["x"][:]
    y_coords = f["y"][:]
    simulation_params = f["params"][:]

print(f"Dataset loaded successfully: {temperature.shape}")
print(f"Number of cases: {temperature.shape[0]}")
print(f"Time steps: {temperature.shape[1]}")
print(f"Grid resolution: {temperature.shape[2]}x{temperature.shape[3]}")

Dataset loaded successfully: (8, 40, 25, 25)
Number of cases: 8
Time steps: 40
Grid resolution: 25x25


In [21]:
# Flatten the 4D data into 2D arrays for training
# Each row represents a single point in space-time

feature_list = []
target_list = []

num_cases = temperature.shape[0]

for case_idx in range(num_cases):
    # Get the hot wall temperature for this case
    hot_temp = simulation_params[case_idx, 0]
    
    # Loop through all time steps and spatial points
    for t_idx in range(temperature.shape[1]):
        for y_idx in range(temperature.shape[2]):
            for x_idx in range(temperature.shape[3]):
                
                # Input features: x, y, time, boundary condition
                features = [
                    x_coords[x_idx],
                    y_coords[y_idx],
                    time_points[t_idx],
                    hot_temp
                ]
                
                # Output targets: T, lf, ux, uy
                targets = [
                    temperature[case_idx, t_idx, y_idx, x_idx],
                    liquid_fraction[case_idx, t_idx, y_idx, x_idx],
                    velocity_x[case_idx, t_idx, y_idx, x_idx],
                    velocity_y[case_idx, t_idx, y_idx, x_idx]
                ]
                
                feature_list.append(features)
                target_list.append(targets)

# Convert to numpy arrays
X = np.array(feature_list, dtype=np.float32)
y = np.array(target_list, dtype=np.float32)

print(f"Input features shape: {X.shape}")
print(f"Output targets shape: {y.shape}")
print(f"Total training samples: {X.shape[0]:,}")

Input features shape: (200000, 4)
Output targets shape: (200000, 4)
Total training samples: 200,000


In [22]:
# Normalize inputs and outputs using z-score normalization
# This helps the neural network train more effectively

X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8  # small epsilon to avoid division by zero

y_mean = y.mean(axis=0)
y_std = y.std(axis=0) + 1e-8

# Apply normalization
X_normalized = (X - X_mean) / X_std
y_normalized = (y - y_mean) / y_std

print("Data normalized successfully")
print(f"Input range: [{X_normalized.min():.2f}, {X_normalized.max():.2f}]")
print(f"Output range: [{y_normalized.min():.2f}, {y_normalized.max():.2f}]")
print(f"\nOutput standard deviations: {y_std}")

Data normalized successfully
Input range: [-1.69, 1.69]
Output range: [-11.16, 17.84]

Output standard deviations: [8.7390232e+00 3.7840670e-01 4.1764224e-04 4.2380349e-04]


In [23]:
from sklearn.model_selection import train_test_split

# Split data into training and validation sets (80-20 split)
X_train, X_val, y_train, y_val = train_test_split(
    X_normalized, y_normalized, 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Validation samples: {X_val.shape[0]:,}")

Training samples: 160,000
Validation samples: 40,000


In [24]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Enable performance optimizations
    torch.backends.cudnn.benchmark = True

# Convert numpy arrays to PyTorch tensors and move to GPU
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)

print("Data loaded to GPU successfully")

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Data loaded to GPU successfully


In [25]:
class SinusoidalLayer(nn.Module):
    """
    Single layer with sinusoidal activation (SIREN architecture)
    Paper: Implicit Neural Representations with Periodic Activation Functions
    """
    def __init__(self, input_dim, output_dim, omega=30.0, is_first_layer=False):
        super().__init__()
        self.input_dim = input_dim
        self.omega = omega
        self.linear = nn.Linear(input_dim, output_dim)
        self.is_first_layer = is_first_layer
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Custom weight initialization for SIREN networks"""
        with torch.no_grad():
            if self.is_first_layer:
                bound = 1.0 / self.input_dim
            else:
                bound = np.sqrt(6.0 / self.input_dim) / self.omega
            
            self.linear.weight.uniform_(-bound, bound)
    
    def forward(self, x):
        return torch.sin(self.omega * self.linear(x))


class PCMNetwork(nn.Module):
    """
    Neural network for PCM phase change prediction
    Uses SIREN architecture for smooth representation of physical fields
    """
    def __init__(self, hidden_size=512, num_layers=8):
        super().__init__()
        
        # Build the network
        layers = []
        
        # First layer (4 inputs: x, y, t, T_hot)
        layers.append(SinusoidalLayer(4, hidden_size, is_first_layer=True))
        
        # Hidden layers
        for _ in range(num_layers):
            layers.append(SinusoidalLayer(hidden_size, hidden_size))
        
        self.hidden_layers = nn.ModuleList(layers)
        
        # Output layer (4 outputs: T, lf, ux, uy)
        self.output_layer = nn.Linear(hidden_size, 4)
        
        # Initialize output layer
        with torch.no_grad():
            bound = np.sqrt(6.0 / hidden_size) / 30.0
            self.output_layer.weight.uniform_(-bound, bound)
    
    def forward(self, x):
        # Pass through all hidden layers
        for layer in self.hidden_layers:
            x = layer(x)
        
        # Final linear output
        output = self.output_layer(x)
        return output

print("Model architecture defined")

Model architecture defined


In [26]:
# Create PyTorch datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create data loaders for batching
batch_size = 8192
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Data loaders ready (batch size: {batch_size})")

Data loaders ready (batch size: 8192)


In [19]:
import torch.nn.functional as F

def train_single_model(epochs=1000, learning_rate=1e-4):
    """Train a single PCM prediction model"""
    
    # Initialize model
    model = PCMNetwork(hidden_size=512, num_layers=8).to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")
    
    # Setup optimizer and learning rate scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    lr_milestones = [50, 100, 200, 400, 600, 800]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=lr_milestones, gamma=0.5
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    max_patience = 100
    
    training_history = []
    validation_history = []
    
    print("\nStarting training...\n")
    
    for epoch in range(1, epochs + 1):
        
        # Training phase
        model.train()
        train_loss_sum = 0.0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = F.mse_loss(predictions, batch_y)
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss_sum += loss.item()
        
        avg_train_loss = train_loss_sum / len(train_loader)
        scheduler.step()
        
        # Validation phase (every 10 epochs)
        if epoch % 10 == 0:
            model.eval()
            val_loss_sum = 0.0
            output_losses = torch.zeros(4)
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    predictions = model(batch_X)
                    val_loss_sum += F.mse_loss(predictions, batch_y).item()
                    
                    # Track individual output losses
                    for i in range(4):
                        output_losses[i] += F.mse_loss(
                            predictions[:, i].cpu(), 
                            batch_y[:, i].cpu()
                        ).item()
            
            avg_val_loss = val_loss_sum / len(val_loader)
            output_losses /= len(val_loader)
            
            training_history.append(avg_train_loss)
            validation_history.append(avg_val_loss)
            
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch:4d} | Train: {avg_train_loss:.5f} | Val: {avg_val_loss:.5f} | LR: {current_lr:.1e}", end="")
            
            # Print detailed breakdown every 50 epochs
            if epoch % 50 == 0:
                print()
                print(f"         T: {output_losses[0]:.5f} | lf: {output_losses[1]:.5f} | ux: {output_losses[2]:.5f} | uy: {output_losses[3]:.5f}")
            
            # Save best model
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                torch.save(model.state_dict(), "best_model.pt")
                print(" ✓ saved")
            else:
                patience_counter += 1
                print()
            
            # Early stopping
            if patience_counter >= max_patience:
                print(f"\nEarly stopping triggered after {epoch} epochs")
                break
    
    print(f"\nTraining complete. Best validation loss: {best_val_loss:.5f}")
    return model, best_val_loss

# Train the model
trained_model, final_loss = train_single_model(epochs=1000)


ENSEMBLE FROM BEST MODEL - GUARANTEED ≥0.162

Creating Model 1/5 (fine-tuned from best)
  Starting from: 0.16244
  E020 | Val: 0.16200 ✅
  E040 | Val: 0.16151 ✅
  E060 | Val: 0.16136 ✅
  E080 | Val: 0.16134 ✅
  E100 | Val: 0.16124 ✅
  E120 | Val: 0.16117 ✅
  E140 | Val: 0.16115 ✅
  E160 | Val: 0.16121
  E180 | Val: 0.16117
  E200 | Val: 0.16123
  Final: 0.16115

Creating Model 2/5 (fine-tuned from best)
  Starting from: 0.16244
  E020 | Val: 0.16161 ✅
  E040 | Val: 0.16120 ✅
  E060 | Val: 0.16100 ✅
  E080 | Val: 0.16092 ✅
  E100 | Val: 0.16072 ✅
  E120 | Val: 0.16060 ✅
  E140 | Val: 0.16056 ✅
  E160 | Val: 0.16051 ✅
  E180 | Val: 0.16054
  E200 | Val: 0.16052
  Final: 0.16051

Creating Model 3/5 (fine-tuned from best)
  Starting from: 0.16244
  E020 | Val: 0.16064 ✅
  E040 | Val: 0.16042 ✅
  E060 | Val: 0.16002 ✅
  E080 | Val: 0.15993 ✅
  E100 | Val: 0.16002
  E120 | Val: 0.15988 ✅
  E140 | Val: 0.15973 ✅
  E160 | Val: 0.15969 ✅
  E180 | Val: 0.15971
  E200 | Val: 0.15974
  Final: 0.1